# Fig. 7 and S7: AML mosaic integration with mutation heads

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/reproducibility/api/fig7_aml.ipynb)

Paired AML CITE-seq (Knorr et al. 2023) trains the bridge. Two cohorts that were never measured together are projected into it: van Galen et al. (2019) scRNA-seq through the RNA encoder, and DAb-seq (Demaree et al. 2021) protein through the protein encoder. Binary mutation heads are then refined on the cells that carry genotype calls in those cohorts, with missing calls masked per gene. Settings follow the archived notebook `UniVI_manuscript_GR-Figure__7__AML_bridge_mapping_and_fine-tuning.ipynb`; preprocessing uses UniVI's fitted preprocessors, and the refinement uses `UniVIRefiner`, so results will be close to, not identical with, the published ones.

As in the article, head evaluation uses a cell-level hold-out, so it measures generalization to new cells from the same cohorts, not to new patients.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.1"

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.metrics import average_precision_score, roc_auc_score

import univi.datasets as uds
from univi import (ClassHeadConfig, ModalityConfig, RefinementConfig, TrainingConfig, UniVIConfig,
                   UniVIMultiModalVAE, UniVIRefiner, UniVITrainer, predict_heads_adata)
from univi.preprocessing import ADTPreprocessor, RNAPreprocessor
from univi.utils.seed import set_seed
from univi.workflows import make_loader, stack_embeddings

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)

In [ ]:
N_EPOCHS = 3000        # archived notebook: 3000 with early stopping (patience 150)
REFINE_EPOCHS = 300
GENES = ["NPM1", "DNMT3A", "FLT3", "TP53", "TET2", "IDH2"]
MIN_PER_CLASS = 20     # a head is trained for a gene only with at least this many mutant and wild-type cells

## Data and patient-level split of the bridge

The CITE-seq bridge is split by sample (8 training, 1 validation, 2 test samples; Supplemental Table S7). All RNA genes shared with the van Galen cohort are used, and the 18 protein markers shared with DAb-seq.

In [ ]:
data = uds.load("aml_mosaic")
cite_rna, cite_adt, vg, dab = data["cite_rna"], data["cite_adt"], data["vangalen_rna"], data["dabseq_adt"]
if "split" in cite_rna.obs:
    splits = {k: np.flatnonzero(cite_rna.obs["split"].to_numpy() == k) for k in ("train", "val", "test")}
else:
    samples = np.random.default_rng(0).permutation(cite_rna.obs["sample_id"].astype(str).unique())
    group = cite_rna.obs["sample_id"].astype(str).to_numpy()
    splits = {"test": np.flatnonzero(np.isin(group, samples[:2])), "val": np.flatnonzero(np.isin(group, samples[2:3])),
              "train": np.flatnonzero(np.isin(group, samples[3:]))}
print({k: len(v) for k, v in splits.items()})

rna_prep = RNAPreprocessor(n_hvg=None, scale=True).fit(cite_rna[splits["train"]])
adt_prep = ADTPreprocessor(scale=True, clip=10.0).fit(cite_adt[splits["train"]])
cite = {"rna": rna_prep.transform(cite_rna), "adt": adt_prep.transform(cite_adt)}
vg_pp, dab_pp = rna_prep.transform(vg), adt_prep.transform(dab)

In [ ]:
cfg = UniVIConfig(
    latent_dim=30, beta=1.15, gamma=1.75, encoder_dropout=0.10, decoder_dropout=0.05,
    modalities=[
        ModalityConfig("rna", cite["rna"].n_vars, [1024, 512, 256, 128, 64], [64, 128, 256, 512, 1024], likelihood="gaussian"),
        ModalityConfig("adt", cite["adt"].n_vars, [128, 64, 32], [32, 64, 128], likelihood="gaussian"),
    ],
)
model = UniVIMultiModalVAE(cfg, loss_mode="v1", v1_recon="avg", normalize_v1_terms=True)
UniVITrainer(
    model, make_loader({m: a[splits["train"]] for m, a in cite.items()}, batch_size=256, shuffle=True, drop_last=True),
    make_loader({m: a[splits["val"]] for m, a in cite.items()}, batch_size=1024),
    TrainingConfig(n_epochs=N_EPOCHS, batch_size=256, lr=1e-4, weight_decay=1e-5, device=device,
                   early_stopping=True, patience=150, log_every=100),
).fit();

## Project the cohorts into the bridge

In [ ]:
joint = stack_embeddings(model, [("CITE-seq", "rna", cite["rna"]), ("van Galen", "rna", vg_pp), ("DAb-seq", "adt", dab_pp)],
                         device=device)
sc.pp.neighbors(joint, use_rep="X_univi", n_neighbors=30)
sc.tl.umap(joint, random_state=0)
sc.pl.umap(joint, color=["cohort"], legend_fontsize=8)

## Mutation labels

van Galen cells carry detected mutant and wild-type transcripts as text (`MutTranscripts`, `WtTranscripts`): a cell is mutant for a gene if only mutant transcripts of it were seen, wild-type if only wild-type ones were, and unlabeled otherwise. DAb-seq cells carry per-gene calls in `mut_<GENE>` (1, 0, or missing).

In [ ]:
def vg_labels(obs, gene):
    mut = obs["MutTranscripts"].fillna("").astype(str).str.upper().str.contains(gene, regex=False)
    wt = obs["WtTranscripts"].fillna("").astype(str).str.upper().str.contains(gene, regex=False)
    y = np.full(len(obs), np.nan)
    y[(mut & ~wt).to_numpy()] = 1.0
    y[(wt & ~mut).to_numpy()] = 0.0
    return y

labels = {"vg": {g: vg_labels(vg_pp.obs, g) for g in GENES},
          "dab": {g: (dab_pp.obs[f"mut_{g}"].astype(float).to_numpy() if f"mut_{g}" in dab_pp.obs
                      else np.full(dab_pp.n_obs, np.nan)) for g in GENES}}
counts = pd.DataFrame({(c, s): {g: int(np.nansum(labels[c][g] == v)) for g in GENES}
                       for c in labels for s, v in [("mut", 1.0), ("wt", 0.0)]})
heads = [g for g in GENES if (counts.loc[g, [(c, "mut") for c in labels]].sum() >= MIN_PER_CLASS
                              and counts.loc[g, [(c, "wt") for c in labels]].sum() >= MIN_PER_CLASS)]
print("heads trained for:", heads)
counts

## Refine mutation heads

One binary head per gene is warmed up on frozen encoders, then trained together with both encoders at a small learning rate, with decoders frozen and a penalty that keeps cells near their original latent positions. 20% of labeled cells in each cohort are held out for evaluation.

In [ ]:
rng = np.random.default_rng(0)
hold = {c: rng.random(len(next(iter(labels[c].values())))) < 0.2 for c in labels}
def y_dict(c, mask):
    return {f"mut_{g}": np.where(mask & np.isfinite(labels[c][g]), labels[c][g], -1).astype(np.float32) for g in heads}

for g in heads:
    model.add_classification_head(ClassHeadConfig(f"mut_{g}", n_classes=2, head_type="binary", hidden_dims=[64, 32],
                                                  dropout=0.1, batchnorm=False, layernorm=True),
                                  label_names=["wt", "mut"])
refiner = UniVIRefiner(
    model,
    [make_loader({"rna": vg_pp}, labels=y_dict("vg", ~hold["vg"]), batch_size=256, shuffle=True),
     make_loader({"adt": dab_pp}, labels=y_dict("dab", ~hold["dab"]), batch_size=256, shuffle=True)],
    [make_loader({"rna": vg_pp}, labels=y_dict("vg", hold["vg"]), batch_size=1024),
     make_loader({"adt": dab_pp}, labels=y_dict("dab", hold["dab"]), batch_size=1024)],
    device=device, encoder_modalities=["rna", "adt"],
    replay_loader=make_loader({m: a[splits["train"]] for m, a in cite.items()}, batch_size=256, shuffle=True, drop_last=True),
    config=RefinementConfig(max_epochs=REFINE_EPOCHS, warmup_epochs=min(50, REFINE_EPOCHS), lr_head=2e-4,
                            lr_encoder=1e-5, latent_weight=1.0, replay_weight=1.0, patience=40, log_every=50),
) if heads else None
if refiner is not None:
    refiner.fit();

In [ ]:
def positive_probability(p):
    p = np.asarray(p)
    return p.reshape(len(p), -1)[:, -1]

rows, prob = [], {}
for c, adata, mod in [("vg", vg_pp, "rna"), ("dab", dab_pp, "adt")]:
    out = predict_heads_adata(model, adata, mod, device=device) if heads else {}
    for g in heads:
        p = positive_probability(out[f"mut_{g}"])
        prob[(c, g)] = p
        y = labels[c][g]
        m = hold[c] & np.isfinite(y)
        if m.sum() and len(np.unique(y[m])) == 2:
            rows.append({"cohort": c, "gene": g, "n_eval": int(m.sum()),
                         "AUC": roc_auc_score(y[m], p[m]), "AP": average_precision_score(y[m], p[m])})
pd.DataFrame(rows).round(3)

Predicted mutation probabilities over all embedded cells (Supplemental Fig. S7D–I):

In [ ]:
for g in heads:
    joint.obs[f"p_{g}"] = np.concatenate([np.full(cite["rna"].n_obs, np.nan), prob[("vg", g)], prob[("dab", g)]])
if heads:
    sc.pl.umap(joint, color=[f"p_{g}" for g in heads], ncols=3, cmap="viridis", vmin=0, vmax=1)